(course-core-05)=

# Module 5: Combined Forms

**Welcome back, Apprentice Master.** In Module 1, Module 2, and Module 3, you learned how MolSysMT handles individual file formats, attributes, and native objects.

However, real computational workflows often split a system's data across multiple complementary **Forms**. For example, a heavy molecular dynamics simulation might store its structural topology in an H5MSM or PDB file, while storing thousands of coordinate frames in a binary DCD or XTC trajectory file.

Traditionally, loading a topology from one file format and a trajectory from another required complex merging scripts or third-party wrappers. In MolSysMT, you simply pass a Python list of complementary forms, and the framework virtually merges them into a single, unified molecular system.

```{admonition} Glossary: Combined Forms
:class: dropdown info
**Combined Forms** (or multi-forms) refers to MolSysMT's ability to treat a list of complementary data forms (e.g., a topology file and a trajectory file) as a single, virtual molecular system without requiring manual array merging or file duplication.
```

(course-core-05-learning-outcomes)=
```{admonition} Learning Outcomes
:class: dropdown learning-outcomes

By the end of this module, you will be able to:
- Combine separate topology and trajectory forms into a single virtual system list.
- Validate multi-form lists using `msm.is_a_molecular_system()`.
- Trace attribute origins and precedence rules using `msm.where_is_attribute()`.
- Extract data seamlessly across multi-forms using `msm.get()`.
- Consolidate multi-form lists into a single unified native form using `msm.convert()`.
```

### 1. The Virtual Multi-Form System

Let's begin by loading the topology and trajectory files of our solvated **Chicken Villin HP35** demonstration system.

In [1]:
import molsysmt as msm
from molsysmt import systems

# Load topology file (H5MSM form) and trajectory file (DCD form)
topology_file = systems['chicken villin HP35']['chicken_villin_HP35_solvated.h5msm']
trajectory_file = systems['chicken villin HP35']['traj_chicken_villin_HP35_solvated.dcd']

# Combine them into a single multi-form list
composite_system = [topology_file, trajectory_file]

print(f"Is it a valid system? {msm.is_a_molecular_system(composite_system)}")
print(f"Canonical form: {msm.get_form(composite_system)}")

Is it a valid system? True
Canonical form: ['file:h5msm', 'file:dcd']


:::{hint}
:class: dropdown
**msm.is_a_molecular_system()**: Validates whether a single item or list of items forms a valid, complementary molecular system. See API doc: {func}`molsysmt.basic.is_a_molecular_system`.
:::

### 2. Attribute Precedence: `msm.where_is_attribute()`

When a multi-form list contains complementary items, MolSysMT routes queries to whichever item provides the required data. 

If two items in your list provide the same attribute (for instance, if both the H5MSM topology and DCD trajectory contain coordinates), MolSysMT follows a simple **Rule of Precedence**: the **first** item in the list providing that attribute takes priority.

We can inspect which form provides a specific attribute using **`msm.where_is_attribute()`**:

In [2]:
# Trace the origin of 'atom_name'
source_names = msm.where_is_attribute(composite_system, 'atom_name')
print(f"Atom names are coming from: {source_names}")

# Trace the origin of 'coordinates'
source_coords = msm.where_is_attribute(composite_system, 'coordinates')
print(f"Coordinates are coming from: {source_coords}")

Atom names are coming from: ('/home/diego/repos@uibcdf/molsysmt/molsysmt/data/h5msm/chicken_villin_HP35_solvated.h5msm', 'file:h5msm')
Coordinates are coming from: ('/home/diego/repos@uibcdf/molsysmt/molsysmt/data/dcd/traj_chicken_villin_HP35_solvated.dcd', 'file:dcd')


:::{hint}
:class: dropdown
**msm.where_is_attribute()**: Identifies which item in a multi-source list provides a specific attribute. See API doc: {func}`molsysmt.basic.where_is_attribute`.
:::

### 3. Querying Attributes Across Multi-Forms: `msm.get()`

Because MolSysMT abstracts multi-form lists as unified systems, you can extract topological and structural attributes simultaneously using **`msm.get()`**.

Spatial coordinates extracted from multi-forms maintain the invariant **3D array shape** `(n_structures, n_atoms, 3)` across all trajectory frames:

In [3]:
# Extract atom names and coordinates for the first 5 atoms across the trajectory
names, coords = msm.get(composite_system, selection=[0, 1, 2, 3, 4], atom_name=True, coordinates=True)

print(f"Atom names: {names}")
print(f"Coordinates array shape: {coords.shape} (n_structures, n_atoms, spatial:x,y,z)")

# Y coordinate (index 1) of the 3rd atom (index 2) in the 4th structure (index 3)
print(f"Y coordinate of 3rd atom in 4th frame: {coords[3, 2, 1]}")

Atom names: ['C', 'O', 'CH3', 'H1', 'H2']
Coordinates array shape: (20, 5, 3) (n_structures, n_atoms, spatial:x,y,z)
Y coordinate of 3rd atom in 4th frame: 1.0159252166748045 nanometer


```{admonition} Terminology Note: "Structure" vs. "Frame"
:class: dropdown note
In MolSysMT, the canonical term for a 3D spatial conformation is **`structure`** (hence `n_structures` and array shape `(n_structures, n_atoms, 3)`). While time-resolved molecular dynamics simulations commonly refer to each time step as a *frame*, a molecular system can also represent NMR conformational ensembles or non-sequential structural states without time dependence. In trajectory contexts, calling stored structures *frames* is natural, but `structure` remains MolSysMT's universal technical term.
```

### 4. Consolidating Multi-Forms into a Single Form: `msm.convert()`

While working with multi-form lists is convenient, you will often want to consolidate separate topology and trajectory files into a single, unified native object or disk file.

You can convert a multi-form list into a single form with a single call to **`msm.convert()`**:

In [4]:
# Consolidate the multi-form list into a single native MolSys object
unified_molsys = msm.convert(composite_system, to_form='molsysmt.MolSys')

print(f"New consolidated form: {msm.get_form(unified_molsys)}")
print(f"Number of structures in consolidated system: {msm.get(unified_molsys, n_structures=True)}")

dcdplugin) detected standard 32-bit DCD file of native endianness
dcdplugin) CHARMM format DCD file (also NAMD 2.1 and later)
dcdplugin) detected standard 32-bit DCD file of native endianness
dcdplugin) CHARMM format DCD file (also NAMD 2.1 and later)
dcdplugin) detected standard 32-bit DCD file of native endianness
dcdplugin) CHARMM format DCD file (also NAMD 2.1 and later)
dcdplugin) detected standard 32-bit DCD file of native endianness
dcdplugin) CHARMM format DCD file (also NAMD 2.1 and later)
dcdplugin) detected standard 32-bit DCD file of native endianness
dcdplugin) CHARMM format DCD file (also NAMD 2.1 and later)
dcdplugin) detected standard 32-bit DCD file of native endianness
dcdplugin) CHARMM format DCD file (also NAMD 2.1 and later)
dcdplugin) detected standard 32-bit DCD file of native endianness
dcdplugin) CHARMM format DCD file (also NAMD 2.1 and later)
dcdplugin) detected standard 32-bit DCD file of native endianness
dcdplugin) CHARMM format DCD file (also NAMD 2.1 and

New consolidated form: molsysmt.MolSys
Number of structures in consolidated system: 20


--- 

### 🏆 Challenge 5: The System Weaver

1. Load a PDB ID (`'pdb_id:181L'`) and instantiate an empty native object (`msm.native.MolSys()`).
2. Pass them together in a list: `[msm.native.MolSys(), 'pdb_id:181L']`.
3. Use `msm.is_a_molecular_system()` to verify if the list is recognized as a valid molecular system.
4. Use `msm.convert()` to consolidate the list into a single native `molsysmt.MolSys` object.

Mastering multi-form systems allows you to combine static structural PDB files with heavy trajectory files seamlessly. In [Module 6: Visualizing Anything](../00_Common_Core/06_Visualizing_Anything.ipynb), we will learn how to render interactive 3D visualizations of any system.

```{key-takeaway}
MolSysMT allows you to virtually merge multiple data forms into a single system by passing them as a list, and to consolidate them into a single native object with `msm.convert()`.
```

(course-core-05-see-also)=
:::{seealso}
:class: dropdown
**API Documentation for Functions in this Module:**
- {func}`molsysmt.basic.is_a_molecular_system` — Molecular system validation engine.
- {func}`molsysmt.basic.where_is_attribute` — Multi-source attribute origin inspector.
- {func}`molsysmt.basic.convert` — Form conversion and consolidation engine.

**Related Course Modules & Guides:**
- Previous Module: [Module 4: Native Forms](../00_Common_Core/04_Native_Forms.ipynb)
- Next Module: [Module 6: Visualizing Anything](../00_Common_Core/06_Visualizing_Anything.ipynb)
- User Guide: {ref}`user-foundations`
:::